In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common
from pyspark.sql import functions as F, Window

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
cat = cfg["catalog"]
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print(("PASS " if ok else "FAIL ") + name + (f" -> {detail}" if detail and not ok else ""))

g = lambda t: spark.table(f"{cat}.gold.{t}")
print(cat, "gold dims")

In [0]:
DIMS = {"dim_branch": ("branch_key", "branch_id"), "dim_product": ("product_key", "product_id"),
        "dim_customer": ("customer_key", "customer_id"), "dim_loan": ("loan_key", "loan_id")}

for t, (key, _id) in DIMS.items():
    df = g(t)
    print(f"INFO {t}: {df.count()} rows")
    check(f"{t}: key filled and unique", df.filter(F.col(key).isNull()).count() == 0
          and df.select(key).distinct().count() == df.count())
    check(f"{t}: exactly one unknown member (-1)", df.filter(F.col(key) == -1).count() == 1)

for t, (key, _id) in [("dim_product", DIMS["dim_product"]), ("dim_customer", DIMS["dim_customer"])]:
    df = g(t).filter(F.col(key) != -1)
    multi = df.filter("is_current").groupBy(_id).count().filter("count <> 1").count()
    none = df.select(_id).distinct().count() - df.filter("is_current").select(_id).distinct().count()
    check(f"{t}: one current version per id", multi == 0 and none == 0, f"multi {multi}, none {none}")
    w = Window.partitionBy(_id).orderBy("valid_from")
    overlap = df.withColumn("_next", F.lead("valid_from").over(w)).filter("_next < valid_to").count()
    check(f"{t}: versions do not overlap", overlap == 0, overlap)
    print(f"INFO {t}: current {df.filter('is_current').count()} (plus unknown)")

cust = g("dim_customer")
bad = [c for c in ["full_name", "phone", "aadhaar", "dob", "monthly_income"] if c in cust.columns]
check("dim_customer: no personal columns", not bad, bad)
check("dim_customer: master_customer_id always filled", cust.filter("master_customer_id IS NULL").count() == 0)
print("INFO distinct master_customer_id:", cust.filter("customer_key <> -1").select("master_customer_id").distinct().count())

m = [r[0] for r in spark.sql(f"""SELECT mask_name FROM {cat}.information_schema.column_masks
        WHERE table_schema = 'gold' AND table_name = 'dim_customer' AND column_name = 'id_fingerprint'""").collect()]
check("dim_customer.id_fingerprint masked with mask_token", m == [f"{cat}.ops.mask_token"], m)
check("dim_loan: dpd_bucket always filled", g("dim_loan").filter("dpd_bucket IS NULL").count() == 0)

print(f"\nt18_gold_dims_fqf: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t18_gold_dims_fqf failed")